# Agentic VQA Pipeline — Qwen3-VL 8B

Notebook dedicato all'esperimento con **Qwen3-VL 8B** (`qwen3-vl:8b`) sul benchmark DUDE.

Utilizza il profilo `qwen3vl_focused` ottimizzato in base ai risultati di `metrics_summary_1.csv`:
- **Layout v4** (QUR=0.8824) per cause spaziali e layout
- **DocEl CoT v3** (QUR=0.7112) per cause di struttura documento
- **NLP List OCR** (QUR=0.6310) per cause entità/valori
- NLP Tag evitato (Error Rate 6.42%)

In **Notebook options** abilita:
- Accelerator: **GPU T4 x2**
- Internet: **On**

Aggiungi come Kaggle Dataset il repository e i dati DUDE.

In [ ]:
# ---- PARAMETRI ----
PROJECT_SOURCE = ""  # /kaggle/input/agentic-vqa-pipeline
GIT_REPOSITORY_URL = "https://github.com/matteo-petrelli/Agentic-VQA-Pipeline"  # URL Git pubblico del repository
GIT_REF = "main"

INPUT_JSON_PATH = "/kaggle/input/datasets/matteopetrelli/dude-mixed/DUDE_fixed.json"
IMAGE_DIR = "/kaggle/input/datasets/matteopetrelli/dude-train/content/DUDE_train-val-test_binaries/images/train"
OUTPUT_JSON_PATH = "/kaggle/working/unanswerability_diagnostic_results_qwen3vl8b.json"

OLLAMA_MODEL = "qwen3-vl:8b"
EVIDENCE_GPU = 0
VLM_GPU = 1
ALLOW_SINGLE_GPU_FALLBACK = True
SAMPLING_PERCENTAGE = 0.1
MAX_DOCUMENT_MB = 100
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200

## 1. Configurazione dell'ambiente Kaggle

Individua il progetto, installa le dipendenze e configura `HF_TOKEN` dai Kaggle Secrets.

In [ ]:
# Generated by GitHub Copilot - Aug-31-2026
from pathlib import Path
import subprocess
import sys

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()
_clone_dir = WORKING_DIR / "Agentic-VQA-Pipeline"

# Sincronizza il repository prima di importare i moduli Python.
if PROJECT_SOURCE:
    PROJECT_DIR = Path(PROJECT_SOURCE).expanduser().resolve()
    if not (PROJECT_DIR / "kaggle_utils.py").is_file():
        raise FileNotFoundError(f"Progetto non valido: {PROJECT_DIR}")
elif GIT_REPOSITORY_URL:
    if (_clone_dir / ".git").is_dir():
        subprocess.run(
            ["git", "-C", str(_clone_dir), "pull", "--ff-only", "origin", GIT_REF],
            check=True,
        )
    elif not (_clone_dir / "kaggle_utils.py").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_REPOSITORY_URL, str(_clone_dir)],
            check=True,
        )
    PROJECT_DIR = _clone_dir.resolve()
elif (Path.cwd() / "kaggle_utils.py").is_file():
    PROJECT_DIR = Path.cwd().resolve()
else:
    raise FileNotFoundError("Imposta PROJECT_SOURCE o GIT_REPOSITORY_URL.")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from kaggle_utils import setup_environment

setup_environment(PROJECT_DIR)

## 2. Rilevamento e assegnazione delle GPU

PyTorch vede entrambe le GPU. DOTS/GLiNER useranno `cuda:0`; Ollama riceverà solo GPU 1.

In [ ]:
from kaggle_utils import detect_gpus

EVIDENCE_DEVICE, VLM_GPU = detect_gpus(EVIDENCE_GPU, VLM_GPU, ALLOW_SINGLE_GPU_FALLBACK)

## 3. Caricamento motore DOTS e GLiNER

Carica DOTS e GLiNER sulla GPU di elaborazione. Configura il profilo **qwen3vl_focused**.

In [ ]:
# Generated by GitHub Copilot - Aug-31-2026
import json
import config

config.INPUT_JSON_PATH = INPUT_JSON_PATH
config.IMAGE_DIR = IMAGE_DIR
config.OUTPUT_JSON_PATH = OUTPUT_JSON_PATH
config.VLM_BACKEND = "ollama"
config.OLLAMA_VLM = OLLAMA_MODEL
config.OLLAMA_FORCE_JSON = False
config.EVIDENCE_DEVICE = EVIDENCE_DEVICE
config.SAMPLING_PERCENTAGE = SAMPLING_PERCENTAGE
config.VLM_NUM_CTX = 16384
config.VLM_MAX_TOKENS = 1536
config.PROMPT_PROFILE = "qwen3vl_focused"

from diagnostic_agent.engine import DocumentEngine

engine = DocumentEngine()
print("DocumentEngine caricato correttamente.")

## 4. Caricamento del modello VLM (Ollama)

Avvia Ollama sulla GPU dedicata, scarica `qwen3-vl:8b` e lo tiene residente.

In [ ]:
from kaggle_utils import start_ollama, stop_ollama

api_url = start_ollama(VLM_GPU, OLLAMA_MODEL, WORKING_DIR)
config.OLLAMA_URL = api_url
config.OLLAMA_VLM = OLLAMA_MODEL

## 4.1 Verifica del profilo attivo

Conferma che il profilo `qwen3vl_focused` è stato selezionato e mostra la mappa cause → prompt.

In [ ]:
from diagnostic_agent.profiles import resolve_prompt_profile

active_profile = resolve_prompt_profile(OLLAMA_MODEL, config.PROMPT_PROFILE)
print(f"Active profile: {active_profile.name}")
print(f"Answerer prompt: {active_profile.answerer_prompt}")
print(f"Verifier prompt: {active_profile.verifier_prompt}")
print("\nCause → Prompt mapping:")
for cause, prompt in active_profile.cause_prompts.items():
    print(f"  {cause.value:35s} → {prompt}")

## 5. Esperimento completo (con checkpointing e sampling)

Il run completo usa `run_experiments.main()` che include:
- **Checkpointing**: salva il risultato dopo ogni domanda, riprende da dove si era fermato
- **Sampling**: rispetta `SAMPLING_PERCENTAGE` per limitare le domande elaborate
- **Profile**: usa automaticamente `qwen3vl_focused` grazie al MODEL_PROFILE_MAP

In [ ]:
from scripts.run_experiments import main as run_full_experiment

run_full_experiment(model_name=OLLAMA_MODEL, engine=engine)

## 6. Salvataggio dei risultati

Esporta risultati in JSON, CSV e TXT.

In [ ]:
from kaggle_utils import export_results

export_results(
    OUTPUT_JSON_PATH,
    smoke_result=None,
    smoke_question=None,
    smoke_image_paths=None,
    run_full=True,
    working_dir=WORKING_DIR,
)

### Arresto del server

Esegui questa cella solo quando hai terminato. Il server viene comunque arrestato automaticamente alla chiusura del kernel.

In [ ]:
stop_ollama()
print("Server Ollama arrestato.")